In [1]:
%cd /iridisfs/home/lhl1g23/ssm/mamba

from models.dataloaders.speech_commands import create_dataloaders
from models.dataloaders.dataloaders import get_sc
from models.dataloaders.s4_sc import _SpeechCommands
from models.mamba_audio_classifier import MambaAudioClassifier

train, test = get_sc()

/iridisfs/home/lhl1g23/ssm/mamba


/home/lhl1g23/.conda/envs/mamba/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
CUDA extension for structured kernels (Cauchy and Vandermonde multiplication) not found. Install by going to extensions/kernels/ and running `python setup.py install`, for improved speed and memory efficiency. Note that the kernel changed for state-spaces 4.0 and must be recompiled.
Falling back on slow Cauchy and Vandermonde kernel. Install at least one of pykeops or the CUDA extension for better speed and memory efficiency.


In [2]:
X,y = next(iter(test))
X,y = X.to('cuda'),y.to('cuda')
X.shape

torch.Size([32, 16000, 2])

In [3]:
import numpy as np
from IPython.display import Audio

# Example: generate a 440 Hz sine wave for 2 seconds
sr = 16000 # sample rate

sample = 4
Audio(X[sample, :, 0].to('cpu'), rate=sr) 

In [ ]:
from train.train import MambaClassifierTrainer
from omegaconf import OmegaConf
from mamba_ssm.modules.mamba import MambaConfig
import torch

config = OmegaConf.load('models/configs/sc/sc_raw.yaml')
mamba_conf = MambaConfig(**config.mamba)

model = MambaClassifierTrainer.load_from_checkpoint(
    'checkpoints/sc/raw/last.ckpt',
    data="sc",
    mamba_config=config.mamba,
    model_config=config.model
) 
model.eval()
torch.set_grad_enabled(False)

In [38]:
logits = model(X)
pred = logits.argmax(dim=1)

In [39]:
pred

tensor([0, 7, 7, 7, 1, 8, 1, 1, 7, 1, 7, 8, 7, 1, 0, 7, 7, 1, 2, 1, 0, 1, 2, 0,
        0, 8, 7, 7, 7, 7, 7, 1], device='cuda:0')

In [40]:
y

tensor([2, 2, 7, 2, 7, 0, 3, 1, 4, 4, 6, 4, 2, 9, 9, 7, 7, 1, 8, 5, 3, 2, 4, 8,
        6, 7, 4, 8, 2, 9, 5, 8], device='cuda:0')

In [42]:
(pred == y).float().mean()

tensor(0.1562, device='cuda:0')